How to import CDP data and convert from 25 Hz to 1 Hz

In [ ]:
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import datetime
import pathlib
import statistics
import mputil
import shutil
import glob
import os
import re
import math
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from scipy.optimize import curve_fit
import seaborn as sns
from scipy.stats import gaussian_kde
from scipy.integrate import quad

In [ ]:
bin_name = ['CDP_Bin00', 'CDP_Bin01', 'CDP_Bin02', 'CDP_Bin03', 
            'CDP_Bin04', 'CDP_Bin05', 'CDP_Bin06', 'CDP_Bin07', 
            'CDP_Bin08', 'CDP_Bin09', 'CDP_Bin11', 'CDP_Bin12' ,
            'CDP_Bin13', 'CDP_Bin14', 'CDP_Bin15', 'CDP_Bin16', 
            'CDP_Bin17', 'CDP_Bin18', 'CDP_Bin19', 'CDP_Bin20', 
            'CDP_Bin21', 'CDP_Bin22', 'CDP_Bin23', 'CDP_Bin24', 
            'CDP_Bin25', 'CDP_Bin26', 'CDP_Bin27',
            'CDP_Bin28', 'CDP_Bin29']

CDP = []

dates_CDP = ['2022-01-11', '2022-01-12','2022-01-15', '2022-01-18', 
             '2022-01-19', '2022-01-24', '2022-01-26', '2022-01-27',
             '2022-02-01', '2022-02-02', '2022-02-03', '2022-02-05', 
             '2022-02-15', '2022-02-16', '2022-02-19', '2022-02-22',
             '2022-02-26', 
             '2022-03-03', '2022-03-04', 
             '2022-03-13', '2022-03-14', '2022-03-18', '2022-03-22',
             '2022-03-26', '2022-03-28', '2022-03-29', 
             '2022-05-05', '2022-05-10','2022-05-16', '2022-05-17',
             '2022-05-18',
             '2022-05-20','2022-05-21', '2022-05-31', '2022-06-02', 
             '2022-06-03', '2022-06-05','2022-06-07', '2022-06-08', 
             '2022-06-10','2022-06-11','2022-06-13', '2022-06-14',
             '2022-06-17', '2022-06-18']
for date in dates_CDP:

    dataset = {'Date': date, 'Clear Means': [], 'Cloud Means': []}  # Initialize a dataset dictionary
    datestr = date.replace('-', '')
    fname_CDP = sorted(glob.glob(f'/home/disk/eos4/kathem24/activate/data/CDP/2022/csv/ACTIVATE-LARGE-CDP_HU25_{datestr}_R*.csv'), reverse=True)
    
    run = 1
    for file_path in fname_CDP:
        nums_file_paths = len(fname_CDP)

        if date <= ('2022-03-29'):
            df_CDP = pd.read_csv(file_path, skiprows= 72, quoting=csv.QUOTE_NONE)
        elif date >= ('2022-05-05'):
            df_CDP = pd.read_csv(file_path, skiprows= 73, quoting=csv.QUOTE_NONE)
        # print(f"\nFirst two lines of file: {file_path}")
        # print(df_CDP.head(2))
       
        df_CDP.columns = df_CDP.columns.str.strip('"')
        df_CDP.replace([-9999, -9999.00], np.NaN, inplace=True)
        object_columns = df_CDP.select_dtypes(include=['object']).columns
        df_CDP[object_columns] = df_CDP[object_columns].apply(lambda col: col.str.strip('"'))
        if all(bin_ in df_CDP.columns for bin_ in bin_name):
            df_CDP[bin_name] = df_CDP[bin_name].apply(pd.to_numeric, errors='coerce')

        if nums_file_paths==2:
            if run==1:
                df4 = df_CDP 
            elif run==2:
                df5 = df_CDP 
                frames = [df5,df4]
                df_CDP = pd.concat(frames)
                CDP.append(df_CDP)
                # print(f"\nCombined DataFrame for date {date}:")
                # print(df_CDP)
                break
            

        if nums_file_paths ==1:
            # print(f"\nSingle file DataFrame for date {date}:")
            # print(df_CDP)
            CDP.append(df_CDP)

        run = run+1 


In [ ]:
# ##Converting the CDP data file (25 Hz original format) into seperate 1 Hz data file to allow
# ## the leg file and the 2DS to match the leg start and end times in the CDP.
# # Load the original CDP DataFrame 
df_CDP_25Hz = df_CDP.copy()  # Keep the original untouched

# # # Ensure Time_Start is numeric and create a floored column for grouping
df_CDP_25Hz['Time_Start'] = pd.to_numeric(df_CDP_25Hz['Time_Start'], errors='coerce')
df_CDP_25Hz['Time_Start'] = np.floor(df_CDP_25Hz['Time_Start']).astype(int)

# Group by the floored Time_Start and calculate the mean for all columns
# Time_Start_Floored will represent the new time column
CDP_1Hz = (
    df_CDP_25Hz
    .groupby('Time_Start', as_index=False)
    .mean(numeric_only=True)  # Compute mean only for numeric columns
)

#Save to remote server (Olympus)
output_path = '/home/disk/eos4/kathem24/activate/data/CDP/2022/csv/CDP_modified_1Hz_datalegs.csv'
CDP_1Hz.to_csv(output_path, index=False)

print("First 10 Rows:")
print(CDP_1Hz.head(10))
print("\nLast 10 Rows:")
print(CDP_1Hz.tail(10))
print("CDP 1Hz data saved to 'CDP_modified_1Hz_datalegs.csv'")

In [ ]:
# Directory to save the individual .csv files
output_dir = '/home/disk/eos4/kathem24/activate/data/CDP/2022/csv/CDP_1Hz_files/'
os.makedirs(output_dir, exist_ok=True)  # Ensure the directory exists

# Loop through each DataFrame in the CDP list
for i, df_CDP in enumerate(CDP):
    current_date = dates_CDP[i]
    print(f"Processing date: {current_date}")  # Status update for each date

    # Ensure Time_Start is numeric and create a floored column for grouping
    df_CDP['Time_Start'] = pd.to_numeric(df_CDP['Time_Start'], errors='coerce')
    df_CDP['Time_Start'] = np.floor(df_CDP['Time_Start']).astype(int)

    # Group by the floored Time_Start and calculate the mean for all columns
    CDP_1Hz = (
        df_CDP
        .groupby('Time_Start', as_index=False)
        .mean(numeric_only=True)  # Compute mean only for numeric columns
    )

    # Add a column for the date to keep track
    CDP_1Hz['Date'] = current_date

    # Save to a CSV file with the date in the filename
    output_file = os.path.join(output_dir, f"CDP_1Hz_{current_date}.csv")
    CDP_1Hz.to_csv(output_file, index=False)

    # Print the status with the saved file's path
    print(f"File saved for date {current_date} at: {output_file}")

print("\nAll dates processed and saved as separate files.")

In [ ]:
#Check to make sure that the time stamps were floored correctly. 

# Directory where the files are saved
output_dir = '/home/disk/eos4/kathem24/activate/data/CDP/2022/csv/CDP_1Hz_files/'

# Loop through each file in the directory
for file_name in sorted(os.listdir(output_dir)):
    if file_name.endswith('.csv'):  # Ensure we only read CSV files
        file_path = os.path.join(output_dir, file_name)
        # Read the file
        df = pd.read_csv(file_path)
        
        # Print the file name and the last 4 rows
        print(f"File: {file_name}")
        print("Last 4 rows:")
        print(df.tail(4))
        print("\n")

In [ ]:
##Import CDP_1Hz datafiles and combine into a single DF

# Directory where the individual .csv files are saved
output_dir = '/home/disk/eos4/kathem24/activate/data/CDP/2022/csv/CDP_1Hz_files/'

# List all files in the directory
file_list = sorted([f for f in os.listdir(output_dir) if f.endswith('.csv')])

# Initialize an empty list to store DataFrames
dataframes = []

# Loop through each file and read it into a DataFrame
for file_name in file_list:
    file_path = os.path.join(output_dir, file_name)
    print(f"Importing file: {file_name}")  # Status update
    
    # Read the file and append to the list
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Combine all DataFrames into a single DataFrame
df_CDP_1Hz = pd.concat(dataframes, ignore_index=True)

In [ ]:
print("Column names in the combined DataFrame:")
print(df_CDP_1Hz.columns.tolist())

In [ ]:
output_combined_path = '/home/disk/eos4/kathem24/activate/data/CDP/2022/csv/CDP_1Hz_files/CDP_combined_1Hz.csv'
df_CDP_1Hz.to_csv(output_combined_path, index=False)
print(f"Combined data saved at: {output_combined_path}")

In [ ]:
# Directory where the individual .csv files are saved
output_dir = '/home/disk/eos4/kathem24/activate/data/CDP/2022/csv/CDP_1Hz_files/'

# List all files in the directory
file_list = sorted([f for f in os.listdir(output_dir) if f.endswith('.csv')])

# Initialize an empty list to store DataFrames
CDP_1Hz = []

# Loop through each file and read it into a DataFrame
for file_name in file_list:
    file_path = os.path.join(output_dir, file_name)
    print(f"Importing file: {file_name}")  # Status update
    
    # Read the file and append to the list
    df = pd.read_csv(file_path)
    CDP_1Hz.append(df)

print(f"Length of CDP_1Hz after loading: {len(CDP_1Hz)}")